In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# =========================
# 1. Read filtered CBFV feature tables
# =========================

X_train_filtered = pd.read_csv("X_train_filtered.csv")
X_val_filtered = pd.read_csv("X_val_filtered.csv")
X_test_filtered = pd.read_csv("X_test_filtered.csv")

# Remove possible index columns saved from csv
for X in [X_train_filtered, X_val_filtered, X_test_filtered]:
    if "Unnamed: 0" in X.columns:
        X.drop(columns=["Unnamed: 0"], inplace=True)

# =========================
# 2. Read split datasets to get labels and synthesis conditions
# =========================

df_train = pd.read_csv("data_splits/drx_train.csv")
df_val = pd.read_csv("data_splits/drx_val.csv")
df_test = pd.read_csv("data_splits/drx_test.csv")

y_train = df_train["DRX label"].reset_index(drop=True)
y_val = df_val["DRX label"].reset_index(drop=True)
y_test = df_test["DRX label"].reset_index(drop=True)

print("Original CBFV feature shapes:")
print("X_train_filtered:", X_train_filtered.shape)
print("X_val_filtered:", X_val_filtered.shape)
print("X_test_filtered:", X_test_filtered.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

# Check that rows match
assert len(X_train_filtered) == len(df_train), "Train rows do not match."
assert len(X_val_filtered) == len(df_val), "Validation rows do not match."
assert len(X_test_filtered) == len(df_test), "Test rows do not match."

# =========================
# 3. Add synthesis-condition features
# =========================

METHOD_CATEGORIES = [
    "Hydrothermal",
    "Mechanochemical",
    "Molten-salt",
    "Sol-gel",
    "Solid-state"
]

def make_condition_features(df):
    """
    Make numerical synthesis-condition features from Method, T and t.
    Method is converted into one-hot columns.
    T and t are kept as numerical columns.
    """
    condition_df = pd.DataFrame(index=df.index)

    # ---------- Method ----------
    # Keep the original single-column category-code encoding:
    # 0 Hydrothermal, 1 Mechanochemical, 2 Molten-salt,
    # 3 Sol-gel, 4 Solid-state.
    method_numeric = pd.to_numeric(df["Method"], errors="coerce")
    if not method_numeric.isin(range(len(METHOD_CATEGORIES))).all():
        raise ValueError("Method must contain category codes 0-4.")
    condition_df["Method"] = method_numeric.astype(int)

    # ---------- Temperature ----------
    condition_df["T (°C)"] = pd.to_numeric(
        df["T (°C)"],
        errors="coerce"
    )

    # ---------- Time ----------
    condition_df["t (h)"] = pd.to_numeric(
        df["t (h)"],
        errors="coerce"
    )

    return condition_df.reset_index(drop=True)

condition_train = make_condition_features(df_train)
condition_val = make_condition_features(df_val)
condition_test = make_condition_features(df_test)

print("Condition feature columns:")
print(condition_train.columns.tolist())

# Add condition features to CBFV features
X_train_filtered = pd.concat(
    [X_train_filtered.reset_index(drop=True), condition_train],
    axis=1
)

X_val_filtered = pd.concat(
    [X_val_filtered.reset_index(drop=True), condition_val],
    axis=1
)

X_test_filtered = pd.concat(
    [X_test_filtered.reset_index(drop=True), condition_test],
    axis=1
)

# Make sure validation and test have exactly the same columns as train
X_val_filtered = X_val_filtered.reindex(columns=X_train_filtered.columns, fill_value=0)
X_test_filtered = X_test_filtered.reindex(columns=X_train_filtered.columns, fill_value=0)

# Fill missing T/t values using training median
train_medians = X_train_filtered.median(numeric_only=True)

X_train_filtered = X_train_filtered.fillna(train_medians)
X_val_filtered = X_val_filtered.fillna(train_medians)
X_test_filtered = X_test_filtered.fillna(train_medians)

print("After adding synthesis-condition features:")
print("X_train_filtered:", X_train_filtered.shape)
print("X_val_filtered:", X_val_filtered.shape)
print("X_test_filtered:", X_test_filtered.shape)

# Check that synthesis features are really included
condition_check_cols = [
    col for col in X_train_filtered.columns
    if col in ["Method", "T (°C)", "t (h)"]
]

print("Synthesis features included:")
print(condition_check_cols)

# =========================
# 4. Scale features using training set only
# =========================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_filtered)
X_val_scaled = scaler.transform(X_val_filtered)
X_test_scaled = scaler.transform(X_test_filtered)

# Convert scaled arrays back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_filtered.columns)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val_filtered.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_filtered.columns)

print("After scaling:")
print("X_train_scaled:", X_train_scaled.shape)
print("X_val_scaled:", X_val_scaled.shape)
print("X_test_scaled:", X_test_scaled.shape)

In [ ]:
from time import time
from collections import OrderedDict

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC

from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score


RANDOM_SEED = 42

classic_models = OrderedDict({
    "dummy": DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED),
    "logreg": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED),
    "rf": RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED),
    "extra_trees": ExtraTreesClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED),
    "gb": GradientBoostingClassifier(random_state=RANDOM_SEED),
    "ada": AdaBoostClassifier(random_state=RANDOM_SEED),
    "knn": KNeighborsClassifier(),
    "svc": SVC(class_weight="balanced", random_state=RANDOM_SEED),
    "linear_svc": LinearSVC(class_weight="balanced", max_iter=5000, random_state=RANDOM_SEED),
})

results = []
trained_models = {}

for model_name, model in classic_models.items():
    print("Training:", model_name)
    
    start_time = time()
    model.fit(X_train_scaled, y_train)
    fit_time = time() - start_time
    
    y_train_pred = model.predict(X_train_scaled)
    y_val_pred = model.predict(X_val_scaled)
    
    results.append({
        "model_name": model_name,
        "fit_time": fit_time,
        "accuracy_train": accuracy_score(y_train, y_train_pred),
        "accuracy_val": accuracy_score(y_val, y_val_pred),
        "balanced_accuracy_train": balanced_accuracy_score(y_train, y_train_pred),
        "balanced_accuracy_val": balanced_accuracy_score(y_val, y_val_pred),
        "f1_macro_train": f1_score(y_train, y_train_pred, average="macro", zero_division=0),
        "f1_macro_val": f1_score(y_val, y_val_pred, average="macro", zero_division=0),
        "precision_macro_val": precision_score(y_val, y_val_pred, average="macro", zero_division=0),
        "recall_macro_val": recall_score(y_val, y_val_pred, average="macro", zero_division=0),
    })
    
    trained_models[model_name] = model

df_classics = pd.DataFrame(results)

df_classics = df_classics.sort_values("f1_macro_val", ascending=False).reset_index(drop=True)

df_classics

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# =========================
# Save model comparison table
# =========================

df_classics.to_csv("model_comparison_classic_models.csv", index=False)

# =========================
# Choose the best model based on validation macro F1
# =========================

best_model_name = "extra_trees"
best_model = trained_models[best_model_name]

print("Best model used for plots:", best_model_name)

# =========================
# Predict validation set
# =========================

y_val_pred = best_model.predict(X_val_scaled)

# Here 0 = unsuccessful, 1 = successful
labels = [0, 1]

# =========================
# Confusion matrix
# =========================

cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual unsuccessful", "Actual successful"],
    columns=["Predicted unsuccessful", "Predicted successful"]
)

display(cm_df)

# Save confusion matrix for plotting notebook
cm_df.to_csv("confusion_matrix_extra_trees_validation.csv")

# =========================
# Classification report
# =========================

report = classification_report(
    y_val,
    y_val_pred,
    labels=labels,
    target_names=["unsuccessful", "successful"],
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report).T
display(report_df)

report_df.to_csv("classification_report_extra_trees_validation.csv")

# =========================
# Feature importance
# =========================

feature_importance = pd.DataFrame({
    "feature": X_train_filtered.columns,
    "importance": best_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

feature_importance.to_csv("feature_importance_extra_trees.csv", index=False)

# =========================
# Validation predictions
# =========================

val_predictions = df_val.copy()
val_predictions["actual_label"] = y_val.values
val_predictions["predicted_label"] = y_val_pred
val_predictions["correct_prediction"] = (
    val_predictions["actual_label"] == val_predictions["predicted_label"]
)

val_predictions.to_csv("validation_predictions_extra_trees.csv", index=False)

# =========================
# Final evaluation on the held-out test set
# =========================

y_test_pred = best_model.predict(X_test_scaled)
test_cm = confusion_matrix(y_test, y_test_pred, labels=labels)
test_cm_df = pd.DataFrame(
    test_cm,
    index=["Actual unsuccessful", "Actual successful"],
    columns=["Predicted unsuccessful", "Predicted successful"]
)
display(test_cm_df)
test_cm_df.to_csv("confusion_matrix_extra_trees_test.csv")

test_report = classification_report(
    y_test,
    y_test_pred,
    labels=labels,
    target_names=["unsuccessful", "successful"],
    zero_division=0,
    output_dict=True
)
test_report_df = pd.DataFrame(test_report).T
display(test_report_df)
test_report_df.to_csv("classification_report_extra_trees_test.csv")

test_predictions = df_test.copy()
test_predictions["actual_label"] = y_test.values
test_predictions["predicted_label"] = y_test_pred
test_predictions["correct_prediction"] = (
    test_predictions["actual_label"] == test_predictions["predicted_label"]
)
test_predictions.to_csv("test_predictions_extra_trees.csv", index=False)

print("Saved:")
print("model_comparison_classic_models.csv")
print("confusion_matrix_extra_trees_validation.csv")
print("classification_report_extra_trees_validation.csv")
print("feature_importance_extra_trees.csv")
print("validation_predictions_extra_trees.csv")
print("confusion_matrix_extra_trees_test.csv")
print("classification_report_extra_trees_test.csv")
print("test_predictions_extra_trees.csv")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd

# This cell can be run by itself.
# It retrains the Extra Trees model, predicts the 12 experimental samples,
# saves the CSV, then displays the result table below.

env = os.environ.copy()
venv_site_packages = Path(".venv") / "Lib" / "site-packages"

if venv_site_packages.exists():
    current_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(venv_site_packages) + (
        os.pathsep + current_pythonpath if current_pythonpath else ""
    )

result = subprocess.run(
    [sys.executable, "make_extratrees_12_sample_predictions.py"],
    capture_output=True,
    text=True,
    env=env
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Extra Trees prediction failed. See the error message above.")

prediction_results = pd.read_csv(
    "predictions_for_12_experimental_samples_extra_trees_with_conditions.csv"
)

prediction_results = prediction_results.sort_values("Sample").reset_index(drop=True)
prediction_results

In [ ]:
import pandas as pd
from pathlib import Path

prediction_csv = Path("predictions_for_12_experimental_samples_extra_trees_with_conditions.csv")

if "prediction_results" not in globals():
    if not prediction_csv.exists():
        raise FileNotFoundError(
            "Prediction CSV not found. Please run the Extra Trees prediction cell above first."
        )
    prediction_results = pd.read_csv(prediction_csv)

prediction_results_original_order = prediction_results.sort_values("Sample").reset_index(drop=True)

prediction_results_original_order

**Prediction note:** To reproduce the 12-sample Extra Trees prediction, run the standalone prediction cell above. It retrains Extra Trees, writes `predictions_for_12_experimental_samples_extra_trees_with_conditions.csv`, and displays the table.